In [24]:
# Imports
import pandas as pd
import torch
import numpy as np
import json
import os
import random

In [32]:
# Load the JSONL file
file_path = 'cartesia-dataset-dec_10th-hooktheory_18k_melody_cartesia_44k_outputs_v1_with_full_metadata.jsonl'
df = pd.read_json(file_path, lines=True)

# Display the first few rows and basic information about the DataFrame
print("DataFrame Info:")
print(df.info())
print("\nFirst few rows:")
df.head()

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8082 entries, 0 to 8081
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   audio_id               8082 non-null   object
 1   alignments             8082 non-null   object
 2   artist                 8082 non-null   object
 3   alignment_type         8082 non-null   object
 4   supplemental_metadata  8082 non-null   object
dtypes: object(5)
memory usage: 315.8+ KB
None

First few rows:


,audio_id,alignments,artist,alignment_type,supplemental_metadata
0,zngRPKQeoJj,"[{'note': 'C#5', 'token_id': 73, 'start': 0.20...",jonathan-young,refined,"{'tags': ['NO_SWING', 'AUDIO_AVAILABLE', 'USER..."
1,dPoDGJABxnM,"[{'note': 'D4', 'token_id': 62, 'start': 0.256...",adult-swim,user,"{'tags': ['NO_SWING', 'AUDIO_AVAILABLE', 'USER..."
2,nJmBzGwXmAV,"[{'note': 'B4', 'token_id': 71, 'start': 0.0, ...",the-fray,refined,"{'tags': ['NO_SWING', 'AUDIO_AVAILABLE', 'USER..."
3,jDgXneELoKl,"[{'note': 'F3', 'token_id': 53, 'start': 0.0, ...",nirvana,refined,"{'tags': ['NO_SWING', 'AUDIO_AVAILABLE', 'USER..."
4,yvmrddlZxOW,"[{'note': 'C#4', 'token_id': 61, 'start': 0.0,...",mini-pati,refined,"{'tags': ['NO_SWING', 'AUDIO_AVAILABLE', 'USER..."


In [33]:
# Find # of artists in the dataset
unique_artists = df['artist'].unique()
print(f"\nTotal number of entries: {len(df['artist'])}")
print(f"Number of unique artists: {len(unique_artists)}")
print("\nUnique artists:")
print(sorted(unique_artists))



Total number of entries: 8082
Number of unique artists: 2725

Unique artists:
['10cc', '1927', '21-pilots', '2gether', '3-doors-down', '30-seconds-to-mars', '311', '38-special', '3l', '3lau', '3oh3', '40meterp', '4minute', '5-seconds-of-summer', '50-cent', '5sta-family', '98-degrees', '9th-wonder', 'a-boogie-wit-da-hoodie', 'a-day-to-remember', 'a-g-cook', 'a-great-big-world', 'a-ha', 'a-love-like-pi', 'a-perfect-circle', 'a-teens', 'a-tribe-called-quest', 'aage-aleksandersen', 'aaliyah-rose', 'aaron', 'ab-soul', 'abba', 'above-and-beyond', 'above-and-beyond-feat-alex-vargas', 'absofacto', 'absrdst-and-diveo', 'ac-dc', 'ace', 'ace-of-base', 'action-bronson-ft-chance-the-rapper', 'adam-buxton', 'adam-green-and-binki-shapiro', 'adam-lambert', 'adam-neely', 'adam-neely-ft-kate-steinberg', 'adam-routt', 'adele', 'adelle', 'adrian-lux', 'adrianna', 'adrianna-r', 'adult-swim', 'adventure-club', 'adventure-time', 'aerosmith', 'aesop-rock', 'afrojack', 'afrojack-ft-eva-simons', 'afroman', 'ag

In [34]:
# Only print the artists that have more than 1 song in the dataset
artists_with_multiple_songs = df[df.groupby('artist')['artist'].transform('count') > 1]['artist'].unique()
print(f"\nArtists with more than 1 song: {len(artists_with_multiple_songs)}")
print(sorted(artists_with_multiple_songs))



Artists with more than 1 song: 1372
['1927', '21-pilots', '2gether', '3-doors-down', '4minute', '50-cent', '98-degrees', 'a-g-cook', 'a-great-big-world', 'a-ha', 'a-perfect-circle', 'a-teens', 'a-tribe-called-quest', 'aage-aleksandersen', 'aaron', 'abba', 'ac-dc', 'ace', 'ace-of-base', 'adam-buxton', 'adam-green-and-binki-shapiro', 'adam-lambert', 'adam-neely', 'adam-routt', 'adele', 'adult-swim', 'adventure-club', 'adventure-time', 'aerosmith', 'afrojack', 'against-me', 'aimee-mann', 'aina-cook', 'air', 'airplane-mode', 'ajr', 'al-jarreau', 'alabama-3', 'alabama-shakes', 'alan-menken', 'alan-parsons-project', 'alanis-morissette', 'alessia-cara', 'alestorm', 'alex-dezen', 'alex-g', 'alexander-rybak', 'alexandra-stan', 'alice-in-chains', 'alice-merton', 'alicia-keys', 'alison-krauss', 'alizee', 'alkaline-trio', 'all-levels-at-once', 'allie-x', 'alligatoah', 'alphaville', 'alstroemeria-records', 'alt-j', 'alunageorge', 'aly-and-aj', 'am-taxi', 'amber-run', 'amy-diamond', 'amy-winehouse'

In [35]:
# calculate duration of each song

# Calculate duration for all entries
def calculate_duration(metadata):
    try:
        alignment_times = metadata['alignment']['user']['times']
        return alignment_times[-1] - alignment_times[0]  # Using last time minus first time
    except (KeyError, TypeError, IndexError):
        return None  # Return None for entries where we can't calculate duration

# Create a new column with durations
df['duration'] = df['supplemental_metadata'].apply(calculate_duration)

# Print some statistics about the durations
print("\nDuration Statistics (in seconds):")
print(df['duration'].describe())

# Print number of songs in different duration ranges
print("\nNumber of songs in different duration ranges:")
duration_ranges = [
    (0, 30), (30, 60), (60, 120), (120, 180), (180, float('inf'))
]
for start, end in duration_ranges:
    count = len(df[(df['duration'] >= start) & (df['duration'] < end)])
    print(f"{start}-{end} seconds: {count} songs")

# Check if there are any entries where duration couldn't be calculated
null_durations = df['duration'].isnull().sum()
if null_durations > 0:
    print(f"\nWarning: {null_durations} entries have missing duration calculations")


Duration Statistics (in seconds):
count    8082.000000
mean       23.301507
std        11.985182
min         4.870000
25%        15.716711
50%        21.114573
75%        28.767500
max       291.080000
Name: duration, dtype: float64

Number of songs in different duration ranges:
0-30 seconds: 6380 songs
30-60 seconds: 1631 songs
60-120 seconds: 59 songs
120-180 seconds: 8 songs
180-inf seconds: 4 songs


In [36]:
# Find songs less than 10 seconds
short_songs = df[df['duration'] < 3]

# Display information about these short songs
print(f"\nNumber of songs less than 10 seconds: {len(short_songs)}")


Number of songs less than 10 seconds: 0


In [12]:
# First, get artists with multiple songs
artist_counts = df['artist'].value_counts()
artists_with_multiple_songs = artist_counts[artist_counts > 1].index

# Create a sub-dataframe with only songs from artists who have multiple songs
multiple_songs_df = df[df['artist'].isin(artists_with_multiple_songs)]

print(f"\nTotal number of artists with multiple songs: {len(artists_with_multiple_songs)}")
print(f"Total number of songs in this subset: {len(multiple_songs_df)}")



Total number of artists with multiple songs: 127
Total number of songs in this subset: 353


In [28]:
filename = 'vocals.wav'  # Remove the leading slash
base_directory = '/home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/'

def create_filepath_dict(df, base_directory=base_directory, filename=filename):
    """
    Creates a list of dictionaries containing filepath and artist information
    Args:
        df: pandas DataFrame containing the songs
        base_directory: base directory where audio files are stored
    Returns:
        List of dictionaries with filepath and artist information
    """
    filepath_dicts = []
    
    for idx, row in df.iterrows():
        # Use audio_id directly from the DataFrame
        audio_id = row['audio_id']
        
        # Create the full filepath
        filepath = os.path.join(base_directory, audio_id, filename)
        
        # Create dictionary for this entry
        entry_dict = {
            "filepath": filepath,
            "artist": row['artist']
        }
        
        filepath_dicts.append(entry_dict)
    
    return filepath_dicts


# Create filepath dictionaries for the multiple songs dataframe
val_filepaths = create_filepath_dict(df)

# Print some examples to verify
print("\nExample filepath entries:")
for i in range(min(3, len(val_filepaths))):
    print(f"\nEntry {i+1}:")
    print(f"Filepath: {val_filepaths[i]['filepath']}")
    print(f"Artist: {val_filepaths[i]['artist']}")

# Print total number of entries
print(f"\nTotal number of filepath entries created: {len(val_filepaths)}")


Example filepath entries:

Entry 1:
Filepath: /home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/_NgbeZJpmQA/vocals.wav
Artist: rihanna

Entry 2:
Filepath: /home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/Abm_ZrNPoak/vocals.wav
Artist: uni-akiyama

Entry 3:
Filepath: /home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/yvgPkGBAmYq/vocals.wav
Artist: joe-inoue

Total number of filepath entries created: 808


In [29]:
# Create a new dictionary to store the chosen songs
test_songs = {}

# Get list of artists with multiple songs
artists_with_multiple = df['artist'].value_counts()[df['artist'].value_counts() > 1].index

# For each artist with multiple songs
for artist in artists_with_multiple:
    # Get all songs by this artist
    artist_entries = [entry for entry in val_filepaths if entry['artist'] == artist]
    
    if artist_entries:
        # Choose the first song (you could also use random.choice if you want random selection)
        chosen_song = artist_entries[0]
        
        # Add to test_songs dictionary using artist as key
        test_songs[artist] = chosen_song
        
        # Remove the chosen song from val_filepaths
        val_filepaths.remove(chosen_song)

# Print some statistics
print(f"Number of songs moved to test_songs: {len(test_songs)}")
print(f"Number of songs remaining in val_filepaths: {len(val_filepaths)}")

# Print some examples from test_songs
print("\nExample entries in test_songs (one song per artist):")
for i, (artist, song) in enumerate(list(test_songs.items())[:3]):
    print(f"\nEntry {i+1}:")
    print(f"Artist: {artist}")
    print(f"Filepath: {song['filepath']}")

# Print some examples from remaining val_filepaths
print("\nExample entries remaining in val_filepaths:")
for i in range(min(3, len(val_filepaths))):
    print(f"\nEntry {i+1}:")
    print(f"Artist: {val_filepaths[i]['artist']}")
    print(f"Filepath: {val_filepaths[i]['filepath']}")

Number of songs moved to test_songs: 127
Number of songs remaining in val_filepaths: 681

Example entries in test_songs (one song per artist):

Entry 1:
Artist: the-beatles
Filepath: /home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/jDgXpKqNgKl/vocals.wav

Entry 2:
Artist: taylor-swift
Filepath: /home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/d_gwywNQoGV/vocals.wav

Entry 3:
Artist: lady-gaga
Filepath: /home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/yvmrld_vxOW/vocals.wav

Example entries remaining in val_filepaths:

Entry 1:
Artist: uni-akiyama
Filepath: /home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/Abm_ZrNPoak/vocals.wav

Entry 2:
Artist: joe-inoue
Filepath: /home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/yvgPkGBAmYq/vocals.wav

Entry 3:
Artist: metrik
Filepath: /home/sc/gcs-mount/cartes

In [30]:
# Step 1: Add a song by the same artist (comp_1)
for artist, entry in test_songs.items():
    # Extract audio_id from the test song filepath
    test_audio_id = entry['filepath'].split('/')[-2]  # Get the ID from the path
    
    # Find all songs by the same artist in val_filepaths, excluding songs with same audio_id
    same_artist_songs = [song for song in val_filepaths 
                        if song['artist'] == artist 
                        and song['filepath'].split('/')[-2] != test_audio_id]  # Compare audio IDs
    
    if same_artist_songs:  # If there are other songs by the same artist
        # Choose one randomly
        comp_1_song = random.choice(same_artist_songs)
        # Add its filepath to the test_songs entry
        entry['comp_1'] = comp_1_song['filepath']
    else:
        entry['comp_1'] = None  # In case there are no other songs by the same artist

# Step 2: Add a song by a different artist (comp_0)
for artist, entry in test_songs.items():
    # Find all songs by different artists in val_filepaths
    different_artist_songs = [song for song in val_filepaths 
                            if song['artist'] != artist]
    
    if different_artist_songs:  # If there are songs by different artists
        # Choose one randomly
        comp_0_song = random.choice(different_artist_songs)
        # Add its filepath to the test_songs entry
        entry['comp_0'] = comp_0_song['filepath']
    else:
        entry['comp_0'] = None  # In case there are no songs by different artists

# Print the first 3 entries to verify
print("\nFirst 3 entries of test_songs with comparisons:")
for i, (artist, entry) in enumerate(list(test_songs.items())[:3]):
    print(f"\nEntry {i+1}:")
    print(f"Artist: {artist}")
    print(f"Test song filepath: {entry['filepath']}")
    print(f"Test song audio_id: {entry['filepath'].split('/')[-2]}")
    if entry['comp_1']:
        print(f"Comparison song by same artist (comp_1): {entry['comp_1']}")
        print(f"Comp_1 audio_id: {entry['comp_1'].split('/')[-2]}")
    if entry['comp_0']:
        print(f"Comparison song by different artist (comp_0): {entry['comp_0']}")
        print(f"Comp_0 audio_id: {entry['comp_0'].split('/')[-2]}")
    print("-" * 50)

# Print some statistics
print(f"\nTotal number of test songs: {len(test_songs)}")
print(f"Number of test songs with comp_1: {sum(1 for entry in test_songs.values() if entry['comp_1'] is not None)}")
print(f"Number of test songs with comp_0: {sum(1 for entry in test_songs.values() if entry['comp_0'] is not None)}")


First 3 entries of test_songs with comparisons:

Entry 1:
Artist: the-beatles
Test song filepath: /home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/jDgXpKqNgKl/vocals.wav
Test song audio_id: jDgXpKqNgKl
Comparison song by same artist (comp_1): /home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/ZbgORJVkmnY/vocals.wav
Comp_1 audio_id: ZbgORJVkmnY
Comparison song by different artist (comp_0): /home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/nJmBkEYkgAV/vocals.wav
Comp_0 audio_id: nJmBkEYkgAV
--------------------------------------------------

Entry 2:
Artist: taylor-swift
Test song filepath: /home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/d_gwywNQoGV/vocals.wav
Test song audio_id: d_gwywNQoGV
Comparison song by same artist (comp_1): /home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/yvmrlwdKxOW/vocal

In [31]:
# Create the output lines
output_lines = []

# For each test song, create two lines: one for comp_1 and one for comp_0
for artist, entry in test_songs.items():
    # Line for comp_1 (similar artist)
    if entry['comp_1'] is not None:
        output_lines.append(f"1\t{entry['filepath']}\t{entry['comp_1']}")
    
    # Line for comp_0 (different artist)
    if entry['comp_0'] is not None:
        output_lines.append(f"0\t{entry['filepath']}\t{entry['comp_0']}")

# Write to file
output_file = 'comparison_pairs.txt'
with open(output_file, 'w') as f:
    f.write('\n'.join(output_lines))

# Print the first 3 lines of the output file
print("\nFirst 10 lines of the output file:")
for line in output_lines[:10]:
    print(line)


First 10 lines of the output file:
1	/home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/jDgXpKqNgKl/vocals.wav	/home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/ZbgORJVkmnY/vocals.wav
0	/home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/jDgXpKqNgKl/vocals.wav	/home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/nJmBkEYkgAV/vocals.wav
1	/home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/d_gwywNQoGV/vocals.wav	/home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/yvmrlwdKxOW/vocals.wav
0	/home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/d_gwywNQoGV/vocals.wav	/home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody_cartesia_44k_outputs/nLgaAbr_gYp/vocals.wav
1	/home/sc/gcs-mount/cartesia-dataset/dec_10th/hooktheory_18k_melody